# **Retreival pipeline**

In [11]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama

### **Configurations**

In [9]:
db_path = "db/chroma_db"
embedding_model_path = "../../../Data/HuggingFaceEmbeddings Models/all-mpnet-base-v2"

### **Load same embedding model**

In [8]:
embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_path,
    model_kwargs={"device" : "cpu"},
    encode_kwargs={"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### **Connect to existing Vector Database**

In [10]:
# since here Chroma(...) is the constructor for opening or creating a vector store, we passs embedding model as function to use when retreiving

db = Chroma(
    persist_directory=db_path,
    embedding_function=embedding_model
)

In [5]:
retreiver = db.as_retriever(search_kwargs = {"k" : 3})

In [6]:
user_question = "How to apply a loan"

relevant_docs = retreiver.invoke(user_question)

In [7]:
for i, doc in enumerate(relevant_docs, 1):
    print(f"### Document {i} : \n {doc.page_content}\n" + "-"*100 + "\n")

### Document 1 : 
 (b) Mr R Senanayake                                                                                                                                                                A8
      Senkadagala Finance PLC                       Chairman               Loans and advances                                     10,000,000     7,641,745      5,756,347
                                                                           Deposits                                                       –      2,297,029       677,672            A9
                                                                           Debentures                                                     –          6,960           6,960
----------------------------------------------------------------------------------------------------

### Document 2 : 
 Financial               (1) Number and (2) amount of Quantitative           Number,           FN-CB-240 a.1 Table 08 – Number and amount of             96

### **LLM**

In [8]:
combined_docs = "\n".join([f"- {doc.page_content}" for doc in relevant_docs])
combined_docs

'- (b) Mr R Senanayake                                                                                                                                                                A8\n      Senkadagala Finance PLC                       Chairman               Loans and advances                                     10,000,000     7,641,745      5,756,347\n                                                                           Deposits                                                       –      2,297,029       677,672            A9\n                                                                           Debentures                                                     –          6,960           6,960\n- Financial               (1) Number and (2) amount of Quantitative           Number,           FN-CB-240 a.1 Table 08 – Number and amount of             96\n  Inclusion               loans outstanding that qualify                      Presentation                    loans outstanding t

In [9]:
prompt = f"""
    User message :
        {user_question}

    Documents : 
        {combined_docs}

    Instruction : 
        Generate a helpful banking response based on the intent. 
        If you can't find helpfull answer in docs say I'm wasn't able to find any information please contact support.
"""

In [3]:
llm = ChatOllama(
    model="llama3"
)

In [13]:
response = llm.invoke(prompt).content

In [14]:
print(response)

Dear valued customer,

Thank you for considering Senkadagala Finance PLC for your loan needs! We're happy to assist you with the application process.

To apply for a loan, please follow these steps:

1. Gather required documents: You'll need to provide identification and proof of income (if applicable). Our team will guide you through this process.
2. Review our loan options: We offer various loan products designed to support small business and community development. Take a look at our "Lending to priority sectors" figure (A1) for more information on our SME, Exporter, Women Empowerment, Microfinance, and Rural Outreach programs.
3. Contact us: Reach out to our team via phone or email, and we'll schedule an appointment to discuss your loan application. We're available Monday to Friday from 8:30 am to 4:30 pm.

Additionally, please note that we have a dedicated team for loans and advances (A8). You can also review our financial statements (A9) to get a better understanding of our compan

In [4]:
print(llm)

metadata={'versions': {'langchain-core': '1.4.6', 'langchain': '1.3.7'}} output_version=None model='llama3'
